# Tujuan, Hipotesis, dan Data

1. Tujuan:
* Mencari apakah variabel X1 mempengaruhi variabel Y
* Mencari apakah variabel X2 mempengaruhi variabel Y
* Mencari apakah variabel X1 dan X2 secara bersamaan mempengaruhi variabel Y

2. Hipotesis:
* H0 = Tidak ada pengaruh antara X dengan Y
* H1 = Ada pengaruh antara X1 dengan Y
* H2 = Ada pengaruh antara X2 dengan Y
* H3 = Ada pengaruh antara X1 dan X3 secara bersamaan dengan Y

3. Data:
* Data dibuat dengan dummy, tapi menggunakan seed. Jadi akan konsisten walaupun di-run beberapa kali. Masing-masing subbagian X1, X2, dan Y juga memiliki nilai yang didapat dari beberapa pertanyaan dan simulasi jawabannya dengan skala Likert.

Lebih disarankan untuk mempelajari terlebih dahulu `Simple-Regresi Linear.ipynb` dan `Advanced-Regresi Linear X dan Y.ipynb` karena ada panduan lengkap.

# Import dan Create Data

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import pearsonr, shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, linear_rainbow

In [ ]:
# ==========================================
# TAHAP 0: GENERATE DATA "MANUSIAWI"
# ==========================================

# Seed dan n_respondents bisa diganti jika mau mendapatkan data yang berbeda
np.random.seed(21)
n_respondents = 123

# 1. Buat "Karakter" Dasar Responden (Latent Variables)
# X1 dan X2 dibuat independen (berbeda), lalu Y dipengaruhi oleh keduanya
base_X1 = np.random.normal(3.5, 0.8, n_respondents)
base_X2 = np.random.normal(3.8, 0.7, n_respondents)
# Rumus: Y dipengaruhi 45% oleh X1, 40% oleh X2, sisanya faktor lain (noise)
base_Y = (0.45 * base_X1) + (0.40 * base_X2) + np.random.normal(0, 0.4, n_respondents)

# Fungsi membuat jawaban kuesioner 1-5 yang konsisten
def generate_likert(base_score, items=3):
    result = {}
    for i in range(1, items + 1):
        # Tambahkan sedikit variasi agar tidak 100% identik antar pertanyaan
        item_score = base_score + np.random.normal(0, 0.4, n_respondents)
        # Pastikan angka ada di batas 1 sampai 5
        item_score = np.clip(np.round(item_score), 1, 5).astype(int)
        result[i] = item_score
    return result

data = {}

# Buat 3 pertanyaan untuk X1, 3 untuk X2, dan 3 untuk Y
for i, vals in generate_likert(base_X1).items(): data[f'X1_{i}'] = vals
for i, vals in generate_likert(base_X2).items(): data[f'X2_{i}'] = vals
for i, vals in generate_likert(base_Y).items(): data[f'Y_{i}'] = vals

df = pd.DataFrame(data)
df.sample(5)

,X1_1,X1_2,X1_3,X2_1,X2_2,X2_3,Y_1,Y_2,Y_3
94,4,4,4,3,2,3,3,2,3
31,5,5,4,3,4,3,4,4,3
28,3,3,3,4,4,4,3,3,3
40,4,4,3,4,5,4,4,4,4
69,5,4,3,3,2,3,3,3,2


# Uji Validitas dan Reliabilitas

In [ ]:
# ==========================================
# FUNGSI BANTUAN: Menghitung Cronbach's Alpha
# ==========================================
def cronbach_alpha(df_items):
    k = df_items.shape[1]
    var_items = df_items.var(ddof=1).sum()
    var_total = df_items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - (var_items / var_total))

# ==========================================
# TAHAP 1: UJI VALIDITAS & RELIABILITAS
# ==========================================
print("=== 1. UJI VALIDITAS & RELIABILITAS ===")
variabel_dict = {
    'X1 (Misal: Harga)': ['X1_1', 'X1_2', 'X1_3'],
    'X2 (Misal: Pelayanan)': ['X2_1', 'X2_2', 'X2_3'],
    'Y (Misal: Kepuasan)': ['Y_1', 'Y_2', 'Y_3']
}

for nama, kolom in variabel_dict.items():
    skor_total = df[kolom].sum(axis=1)
    print(f"\n--- {nama} ---")

    # Validitas
    for item in kolom:
        r, p = pearsonr(df[item], skor_total)
        print(f"Validitas {item}: r={r:.3f}, p={p:.3f} -> {'Valid' if p < 0.05 and r > 0.3 else 'Tidak Valid'}")

    # Reliabilitas
    alpha = cronbach_alpha(df[kolom])
    print(f"Reliabilitas: Cronbach's Alpha = {alpha:.3f} -> {'Reliabel' if alpha > 0.6 else 'Tidak Reliabel'}")


=== 1. UJI VALIDITAS & RELIABILITAS ===

--- X1 (Misal: Harga) ---
Validitas X1_1: r=0.896, p=0.000 -> Valid
Validitas X1_2: r=0.902, p=0.000 -> Valid
Validitas X1_3: r=0.899, p=0.000 -> Valid
Reliabilitas: Cronbach's Alpha = 0.881 -> Reliabel

--- X2 (Misal: Pelayanan) ---
Validitas X2_1: r=0.868, p=0.000 -> Valid
Validitas X2_2: r=0.844, p=0.000 -> Valid
Validitas X2_3: r=0.872, p=0.000 -> Valid
Reliabilitas: Cronbach's Alpha = 0.825 -> Reliabel

--- Y (Misal: Kepuasan) ---
Validitas Y_1: r=0.851, p=0.000 -> Valid
Validitas Y_2: r=0.870, p=0.000 -> Valid
Validitas Y_3: r=0.873, p=0.000 -> Valid
Reliabilitas: Cronbach's Alpha = 0.830 -> Reliabel


# Agregasi Data

In [ ]:
# ==========================================
# TAHAP 2: AGREGASI DATA
# ==========================================
df_agg = pd.DataFrame()
df_agg['X1'] = df[variabel_dict['X1 (Misal: Harga)']].mean(axis=1)
df_agg['X2'] = df[variabel_dict['X2 (Misal: Pelayanan)']].mean(axis=1)
df_agg['Y']  = df[variabel_dict['Y (Misal: Kepuasan)']].mean(axis=1)
df_agg.sample(5)

,X1,X2,Y
111,4.000000,4.000000,2.333333
120,3.000000,4.000000,1.333333
60,3.666667,4.666667,4.000000
121,4.000000,3.666667,2.666667
119,2.666667,4.000000,2.666667


# Uji Asumsi Klasik

In [ ]:
# ==========================================
# TAHAP 3: UJI ASUMSI KLASIK
# ==========================================
X = df_agg[['X1', 'X2']]
Y = df_agg['Y']
X_const = sm.add_constant(X)
model = sm.OLS(Y, X_const).fit()
residual = model.resid

In [ ]:
# A. Normalitas
stat, p_norm = shapiro(residual)
print(f"1. Normalitas (Shapiro) p: {p_norm:.4f} -> {'Aman' if p_norm > 0.05 else 'Tidak Normal'}")


1. Normalitas (Shapiro) p: 0.6939 -> Aman


In [ ]:
# B. Multikolinearitas
print("2. Multikolinearitas (VIF):")
for i, col in enumerate(X_const.columns):
    if col != 'const':
        vif = variance_inflation_factor(X_const.values, i)
        print(f"   - {col}: {vif:.2f} -> {'Aman' if vif < 10 else 'Terjadi Multikolinearitas'}")

2. Multikolinearitas (VIF):
   - X1: 1.01 -> Aman
   - X2: 1.01 -> Aman


In [ ]:
# C. Heteroskedastisitas
bp_test = het_breuschpagan(residual, X_const)
print(f"3. Heteroskedastisitas p: {bp_test[1]:.4f} -> {'Aman' if bp_test[1] > 0.05 else 'Heteroskedastisitas'}")

3. Heteroskedastisitas p: 0.6357 -> Aman


In [ ]:
# D. Linearitas
rb_stat, rb_p = linear_rainbow(model)
print(f"4. Linearitas (Rainbow) p: {rb_p:.4f} -> {'Aman' if rb_p > 0.05 else 'Tidak Linear'}")

4. Linearitas (Rainbow) p: 0.0897 -> Aman


# Uji Hipotesis

In [ ]:
# ==========================================
# TAHAP 4: UJI HIPOTESIS (REGRESI BERGANDA)
# ==========================================
print("\n=== 3. HASIL UJI HIPOTESIS ===")
print(model.summary())


=== 3. HASIL UJI HIPOTESIS ===
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.452
Model:                            OLS   Adj. R-squared:                  0.443
Method:                 Least Squares   F-statistic:                     49.47
Date:                Tue, 07 Apr 2026   Prob (F-statistic):           2.15e-16
Time:                        06:52:18   Log-Likelihood:                -93.937
No. Observations:                 123   AIC:                             193.9
Df Residuals:                     120   BIC:                             202.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.272

# Jawaban

* X1 memiliki nilai Uji T (P > |t|) 0.000 atau kurang dari 0.05, yang berarti X1 signifikan berpengaruh terhadap Y. Kita menolak H0 dan menerima H1. X1 memiliki koefisien positif, yang berarti jika X1 naik, maka Y juga naik.
* X2 memiliki nilai Uji T (P > |t|) 0.000 atau kurang dari 0.05, yang berarti X2 signifikan berpengaruh terhadap Y. Kita menolak H0 dan menerima H2. X2 memiliki koefisien positif, yang berarti jika X2 naik, maka Y juga naik.
* Hasil Uji F (Prob F-Statistics) 2.15e-16 atau kurang dari 0.05, yang berarti X1 dan X2 secara bersama signifikan berpengaruh terhadap Y. Kita menolak H0 dan menerima H3.
* X1 memiliki pengaruh yang lebih kuat daripada X2 terhadap Y. Hal ini dilihat dari nilai Koefisien X1 > X2 atauu 0.5 > 0.42
* Nilai Adjusted R-Squared adalah 0.443 yang berarti variabel X1 dan X2 dapat menjelaskan 44.3% variasi yang terjadi pada Y. Sisanya 55.6% dijelaskan oleh faktor lain di luar model ini.